In [1]:
import os
import json
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

CORPUS_PATH = "data/raw/corpus.txt"
OUT_DIR     = "tokenizer/"
FINAL_PATH  = os.path.join(OUT_DIR, "tokenizer.json")
VOCAB_SIZE  = 16384
os.makedirs(OUT_DIR, exist_ok=True)

SPECIAL_TOKENS = [
    "<pad>",       # 0
    "<unk>",       # 1
    "<bos>",       # 2
    "<eos>",       # 3
    "<|system|>",  # 4
    "<|user|>",    # 5
    "<|assistant|>", # 6
]

PAD_ID = 0
UNK_ID = 1
BOS_ID = 2
EOS_ID = 3
SYSTEM_ID = 4
USER_ID = 5
ASSISTANT_ID = 6


# ── Step 1: Train ─────────────────────────────────────────────────────────────
print("=" * 60)
print("STEP 1: Training BPE")
print("=" * 60)

tokenizer = Tokenizer(BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
tokenizer.decoder = ByteLevelDecoder()

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=SPECIAL_TOKENS,
    show_progress=True,
)

tokenizer.train([CORPUS_PATH], trainer)
tokenizer.save(FINAL_PATH)
print(f"Saved raw → {FINAL_PATH}  (vocab: {tokenizer.get_vocab_size()} tokens)")

STEP 1: Training BPE



Saved raw → tokenizer_v3/tokenizer.json  (vocab: 16384 tokens)


In [2]:
print("\n--- SENTENCE ROUNDTRIP ---")
for s in [
    "The quick brown fox jumps over the lazy dog",
    "Once upon a time there was a little girl",
    "SELECT name FROM employees WHERE age > 30",
    "CREATE TABLE orders (id INT, name VARCHAR(255))",
    "going inside the organization and looking around",
    "SELECT * FROM orders WHERE id IN (1, 2, 3)",
    "GROUP BY department HAVING COUNT(*) > 5",
    "INSERT INTO users VALUES (1, 'john')",
    "DELETE FROM orders WHERE id = 1",
    "<|user|> show me all orders <|assistant|>",
    "<|user|> SELECT all employees <|assistant|> SELECT * FROM employees",
    "<|user|>A<|assistant|>B",
]:
    ids   = tokenizer.encode(s).ids
    dec   = tokenizer.decode(ids, skip_special_tokens=False)
    match = "✅" if dec.strip() == s.strip() else "❌"
    print(f"  {match} {s}")
    if match == "❌":
        print(f"     got: '{dec}'")
    print(f"     tokens: {len(ids)}")


--- SENTENCE ROUNDTRIP ---
  ✅ The quick brown fox jumps over the lazy dog
     tokens: 9
  ✅ Once upon a time there was a little girl
     tokens: 9
  ✅ SELECT name FROM employees WHERE age > 30
     tokens: 12
  ✅ CREATE TABLE orders (id INT, name VARCHAR(255))
     tokens: 22
  ✅ going inside the organization and looking around
     tokens: 7
  ✅ SELECT * FROM orders WHERE id IN (1, 2, 3)
     tokens: 18
  ✅ GROUP BY department HAVING COUNT(*) > 5
     tokens: 16
  ✅ INSERT INTO users VALUES (1, 'john')
     tokens: 19
  ✅ DELETE FROM orders WHERE id = 1
     tokens: 12
  ✅ <|user|> show me all orders <|assistant|>
     tokens: 7
  ✅ <|user|> SELECT all employees <|assistant|> SELECT * FROM employees
     tokens: 15
  ✅ <|user|>A<|assistant|>B
     tokens: 4


In [3]:
16384 * 384

6291456

In [4]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

# %%
# Load tokenizer
vocab = tokenizer.get_vocab()             # {token_str: id}
id_to_token = {v: k for k, v in vocab.items()}
vocab_size = tokenizer.get_vocab_size()

print(f"Vocab size: {vocab_size}")
print(f"Sample tokens: {list(id_to_token.items())[:10]}")

# %%
# Load MiniLM
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
model.eval()

print("MiniLM loaded")

# %%
# Collect all token surface strings in ID order
# ByteLevel tokens have Ġ (space) prefix — decode to real text
token_strings = []
for i in range(vocab_size):
    surface = id_to_token[i]
    surface = surface.replace("Ġ", " ").replace("Ċ", "\n")
    token_strings.append(surface)

print(f"Total tokens to embed: {len(token_strings)}")
print(f"Sample surfaces: {token_strings[:17]}")   # show special tokens too

# %%
# Embed all tokens in batches
BATCH_SIZE = 256
embeddings = []

for i in range(0, len(token_strings), BATCH_SIZE):
    batch = token_strings[i : i + BATCH_SIZE]
    with torch.no_grad():
        vecs = model.encode(
            batch,
            batch_size=BATCH_SIZE,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
    embeddings.append(vecs)
    print(f"  {min(i + BATCH_SIZE, vocab_size)}/{vocab_size}", end="\r")

embedding_table = np.vstack(embeddings).astype(np.float32)
print(f"\nEmbedding table shape: {embedding_table.shape}")   # (vocab_size, 384)

# %%
# Quick sanity check — similar tokens should be close in embedding space
from numpy.linalg import norm

def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

cat_id  = vocab.get("cat",  vocab.get("Ġcat",  None))
dog_id  = vocab.get("dog",  vocab.get("Ġdog",  None))
king_id = vocab.get("king", vocab.get("Ġking", None))
pad_id  = vocab.get("<pad>")
unk_id  = vocab.get("<unk>")
bos_id  = vocab.get("<bos>")

if cat_id and dog_id:
    sim = cosine_sim(embedding_table[cat_id], embedding_table[dog_id])
    print(f"Similarity cat  <-> dog  : {sim:.4f}  (expect high ~0.7+)")

if cat_id and king_id:
    sim = cosine_sim(embedding_table[cat_id], embedding_table[king_id])
    print(f"Similarity cat  <-> king : {sim:.4f}  (expect lower)")

# Special tokens should be distinct from each other
sim_pad_unk = cosine_sim(embedding_table[pad_id], embedding_table[unk_id])
sim_pad_bos = cosine_sim(embedding_table[pad_id], embedding_table[bos_id])
print(f"Similarity <pad> <-> <unk>: {sim_pad_unk:.4f}  (distinct, not zero)")
print(f"Similarity <pad> <-> <bos>: {sim_pad_bos:.4f}  (distinct, not zero)")

# %%
# Save
np.save("tokenizer/token_embeddings.npy", embedding_table)
print(f"Saved → tokenizer/token_embeddings.npy")
print(f"Size  : {embedding_table.nbytes / 1024 / 1024:.1f} MB")

Vocab size: 16384
Sample tokens: [(1066, 'Ġserv'), (1058, 'Ġsay'), (10327, 'wick'), (6638, 'Ġelim'), (9589, 'Ġlegit'), (13634, 'Ġreferring'), (7295, 'ĠDie'), (544, 'Ġsaw'), (12118, 'Ġcorrel'), (1325, 'ĠHis')]


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MiniLM loaded
Total tokens to embed: 16384
Sample surfaces: ['<pad>', '<unk>', '<bos>', '<eos>', '<|system|>', '<|user|>', '<|assistant|>', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*']
  16384/16384
Embedding table shape: (16384, 384)
Similarity cat  <-> dog  : 0.6606  (expect high ~0.7+)
Similarity cat  <-> king : 0.3610  (expect lower)
Similarity <pad> <-> <unk>: 0.5141  (distinct, not zero)
Similarity <pad> <-> <bos>: 0.5118  (distinct, not zero)
Saved → tokenizer/token_embeddings.npy
Size  : 24.0 MB
